In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/processed/creator_preprocessed.csv"
)

df.head()

,post_id,account_id,account_type,follower_count,media_type,content_category,traffic_source,has_call_to_action,post_datetime,post_date,...,caption_length,hashtags_count,performance_bucket_label,is_weekend,year,month,month_name,quarter,week,time_slot
0,IG0000001,7,brand,3551,reel,Technology,Home Feed,1,2024-11-30 06:00:00,2024-11-30,...,100,7,medium,1,2024,11,November,4,48,Morning
1,IG0000003,15,brand,8167,reel,Beauty,Reels Feed,0,2025-09-11 16:00:00,2025-09-11,...,115,8,high,0,2025,9,September,3,37,Afternoon
2,IG0000004,11,creator,9044,carousel,Music,External,0,2025-09-18 03:00:00,2025-09-18,...,115,7,medium,0,2025,9,September,3,38,Night
3,IG0000005,8,creator,15986,reel,Technology,Profile,0,2025-03-21 09:00:00,2025-03-21,...,112,9,low,0,2025,3,March,1,12,Morning
4,IG0000006,7,brand,3551,carousel,Photography,Reels Feed,1,2025-05-06 08:00:00,2025-05-06,...,111,5,high,0,2025,5,May,2,19,Morning


In [20]:
df["total_engagement"] = (
    df["likes"]
    + df["comments"]
    + df["shares"]
    + df["saves"]
)

In [21]:
df["engagement_per_follower"] = (
    df["total_engagement"]
    / df["follower_count"]
)

In [22]:
df["reach_rate"] = (
    df["reach"]
    / df["follower_count"]
) * 100

In [23]:
df["impression_rate"] = (
    df["impressions"]
    / df["follower_count"]
) * 100

In [24]:
df["virality_score"] = (
    df["shares"]
    / df["reach"].replace(0, np.nan)
) * 100

In [7]:
df["virality_score"] = (
    df["shares"] /
    df["reach"].replace(0, 1)
) * 100

In [25]:
df["save_rate"] = (
    df["saves"]
    / df["reach"].replace(0, np.nan)
) * 100

In [26]:
df["comment_rate"] = (
    df["comments"]
    / df["reach"].replace(0, np.nan)
) * 100

In [27]:
df["like_rate"] = (
    df["likes"]
    / df["reach"].replace(0, np.nan)
) * 100

In [28]:
df["share_rate"] = (
    df["shares"]
    / df["reach"].replace(0, np.nan)
) * 100

In [29]:
df["cta_engagement"] = np.where(
    df["has_call_to_action"] == 1,
    df["total_engagement"],
    0
)

In [31]:
def creator_tier(followers):

    if followers < 10000:
        return "Nano"

    elif followers < 100000:
        return "Micro"

    elif followers < 500000:
        return "Mid"

    elif followers < 1000000:
        return "Macro"

    else:
        return "Mega"

df["creator_tier"] = (
    df["follower_count"]
    .apply(creator_tier)
)

In [32]:
df["creator_tier"].value_counts()

creator_tier
Nano     15436
Micro     7708
Name: count, dtype: int64

In [33]:
df["performance_score"] = (
      df["engagement_rate"] * 0.40
    + df["reach_rate"] * 0.30
    + df["save_rate"] * 0.20
    + df["share_rate"] * 0.10
)

In [34]:
df["content_efficiency"] = (
    df["total_engagement"]
    / df["impressions"].replace(0, np.nan)
) * 100

In [35]:
q1 = df["performance_score"].quantile(0.25)
q3 = df["performance_score"].quantile(0.75)

def performance_category(score):

    if score >= q3:
        return "High"

    elif score <= q1:
        return "Low"

    return "Medium"

df["performance_category"] = (
    df["performance_score"]
    .apply(performance_category)
)

In [36]:
new_features = [
    "total_engagement",
    "engagement_per_follower",
    "reach_rate",
    "impression_rate",
    "virality_score",
    "save_rate",
    "comment_rate",
    "like_rate",
    "share_rate",
    "creator_tier",
    "performance_score",
    "content_efficiency",
    "performance_category"
]

df[new_features].head()

,total_engagement,engagement_per_follower,reach_rate,impression_rate,virality_score,save_rate,comment_rate,like_rate,share_rate,creator_tier,performance_score,content_efficiency,performance_category
0,240,0.067587,121.852999,175.443537,0.161775,0.785764,0.115554,4.483476,0.161775,Nano,36.744630,3.852327,High
1,139,0.017020,20.068569,32.031346,0.061013,1.342282,0.122026,6.955461,0.061013,Nano,6.316368,5.313456,Low
2,98,0.010836,31.811146,35.061920,0.243309,0.000000,0.000000,3.163017,0.243309,Nano,9.580035,3.090508,Low
3,188,0.011760,33.466783,53.190292,0.093458,0.392523,0.149533,2.878505,0.093458,Micro,10.136725,2.210984,Medium
4,179,0.050408,75.133765,97.775275,0.149925,0.862069,0.112444,5.584708,0.149925,Nano,22.748176,5.155530,Medium


In [37]:
df[new_features].describe(include="all")

,total_engagement,engagement_per_follower,reach_rate,impression_rate,virality_score,save_rate,comment_rate,like_rate,share_rate,creator_tier,performance_score,content_efficiency,performance_category
count,23144.000000,23144.000000,23144.000000,23144.000000,23144.000000,23144.000000,23144.000000,23144.000000,23144.000000,23144,23144.000000,23144.000000,23144
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN,NaN,3
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Nano,NaN,NaN,Medium
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15436,NaN,NaN,11572
mean,250.594279,0.037476,72.124253,97.094798,0.216313,0.638492,0.126704,4.321073,0.216313,NaN,21.802517,3.977780,NaN
std,174.699126,0.035470,56.957172,77.200247,0.130357,0.350111,0.087982,2.090711,0.130357,NaN,17.083003,1.950929,NaN
min,0.000000,0.000000,2.689854,3.947204,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,1.003163,0.000000,NaN
25%,114.000000,0.013576,32.714476,43.698273,0.114811,0.357885,0.060140,2.544788,0.114811,NaN,9.983237,2.320378,NaN
50%,207.000000,0.026311,55.216029,74.268641,0.203928,0.624191,0.115875,4.299989,0.203928,NaN,16.727121,3.910877,NaN
75%,352.000000,0.048734,93.026916,125.563343,0.303824,0.902237,0.181758,6.089440,0.303824,NaN,28.079307,5.523322,NaN


In [38]:
df.to_csv(
    "../data/processed/creator_feature_engineered.csv",
    index=False
)